# Minimal quantum-dot example

This notebook reproduces the minimal single-level quantum-dot example from Sec. 3.3 of the QuantumFCS.jl manuscript.

The model is a single electronic level coupled to two reservoirs in the large-bias limit. Electrons enter the dot from the hot reservoir and leave through the cold reservoir. The goal is to compute the first two long-time cumulants of the electron current into the cold reservoir using `QuantumFCS.jl`.

We use `QuantumOptics.jl` to construct the two-state Hilbert space, Hamiltonian, jump operators, and steady state. We then pass the Hamiltonian, jump operators, monitored jump, weights, and steady state to `QuantumFCS.jl`.

In [ ]:
using QuantumOptics
using QuantumFCS

## Model

The quantum dot is represented by a two-state Fock basis: empty state `|0⟩` and occupied state `|1⟩`.

The Hamiltonian is

$$
H = \epsilon_d d^\dagger d .
$$

In the large-bias limit, the relevant jump processes are:

- electron loss into the cold reservoir: $J_{\mathrm{c,loss}} = \sqrt{\kappa_c}\, d$,
- electron gain from the hot reservoir: $J_{\mathrm{h,gain}} = \sqrt{\kappa_h}\, d^\dagger$.

We use the parameters from the manuscript:

$$
\epsilon_d = 1,\qquad \kappa_c = 0.1,\qquad \kappa_h = 0.5.
$$

In [3]:
# Create the basis for the single quantum dot.
# FockBasis(1) contains the states |0⟩ and |1⟩.
basis = FockBasis(1)

# Define lowering and raising operators.
d = destroy(basis)
d_dag = create(basis)

# System parameters.
eps_d = 1.0      # Energy level of the quantum dot.
kappa_c = 0.1    # Coupling strength to the cold reservoir.
kappa_h = 0.5    # Coupling strength to the hot reservoir.

# Hamiltonian H = eps_d * d†d.
H = eps_d * d_dag * d

# Jump operators.
J_c_loss = sqrt(kappa_c) * d
J_h_gain = sqrt(kappa_h) * d_dag

# Full list of jump operators entering the Lindblad equation.
J = [J_c_loss, J_h_gain]

2-element Vector{Operator{FockBasis{Int64}, FockBasis{Int64}, SparseArrays.SparseMatrixCSC{ComplexF64, Int64}}}:
 Operator(dim=2x2)
  basis: Fock(cutoff=1)
      ⋅       0.31622776601683794 + 0.0im
      ⋅                            ⋅     
 Operator(dim=2x2)
  basis: Fock(cutoff=1)
                     ⋅            ⋅     
 0.7071067811865476 + 0.0im       ⋅     

## Steady state

The recursive FCS algorithm requires the stationary state of the Lindblad dynamics. Here we compute it with the steady-state solver from `QuantumOptics.jl`.

In [4]:
# Compute the stationary state of the Lindblad master equation.
rho_ss = steadystate.iterative(H, J)

Operator(dim=2x2)
  basis: Fock(cutoff=1)
 0.166667-0.0im       0.0+0.0im
      0.0+0.0im  0.833333+0.0im

## Monitored current

We monitor the electron-loss jump into the cold reservoir. Following the manuscript, we use the monitored-jump list

$$
mJ = [J_{\mathrm{c,loss}}],
$$

with the weight

$$
\nu = [-1].
$$

The minus sign fixes the counting convention for an electron jumping out of the dot. In the validation below, we compare the magnitude of the mean current with the positive analytical current into the cold reservoir.

In [5]:
# Monitored jump operators.
mJ = [J_c_loss]

# Counting weights.
nu = [-1.0]

# Compute the first two cumulants.
nC = 2

2

## Compute FCS cumulants with the problem-object interface

The `LindbladFCS` object collects the Hamiltonian, jump operators, monitored jumps, weights, steady state, and requested cumulant order. The function `fcscumulants_recursive` then returns the cumulants in ascending order.

In [10]:
# Assemble the FCS problem.
problem = LindbladFCS(;
    H = H,
    J = J,
    mJ = mJ,
    rho_ss = rho_ss,
    nu = nu,
    nC = nC,
)

# Compute current cumulants.
c1, c2 = fcscumulants_recursive(problem)

# The results should be real up to numerical roundoff.
c1 = real(c1)
c2 = real(c2)

println("First cumulant  c₁ = ", c1)
println("Second cumulant c₂ = ", c2)

First cumulant  c₁ = -0.08333333333333336
Second cumulant c₂ = 0.0601851851851852


## Analytical reference

For the large-bias single-level quantum dot, the first two cumulants are

$$
c_1 = \frac{\kappa_c \kappa_h}{\kappa_c + \kappa_h},
$$

and

$$
c_2 =
\frac{\kappa_h^2 + \kappa_c^2}{(\kappa_c + \kappa_h)^2} c_1 .
$$

With $\kappa_c = 0.1$ and $\kappa_h = 0.5$, this gives

$$
c_1 = 0.08333333333333334,\qquad
c_2 = 0.06018518518518519.
$$

In [11]:
# Analytical large-bias reference values.
c1_ref = kappa_c * kappa_h / (kappa_c + kappa_h)
c2_ref = ((kappa_h^2 + kappa_c^2) / (kappa_c + kappa_h)^2) * c1_ref

# The first cumulant may carry the sign implied by the chosen counting convention.
# We therefore validate the current magnitude against the positive reference current.
c1_magnitude_error = abs(abs(c1) - c1_ref)
c2_error = abs(c2 - c2_ref)

println("Reference current magnitude |c₁| = ", c1_ref)
println("Reference noise             c₂  = ", c2_ref)
println("| |c₁_num| - c₁_ref |           = ", c1_magnitude_error)
println("| c₂_num - c₂_ref |             = ", c2_error)

@assert isapprox(abs(c1), c1_ref; rtol = 1e-10, atol = 1e-12)
@assert isapprox(c2, c2_ref; rtol = 1e-10, atol = 1e-12)

Reference current magnitude |c₁| = 0.08333333333333334
Reference noise             c₂  = 0.0601851851851852
| |c₁_num| - c₁_ref |           = 1.3877787807814457e-17
| c₂_num - c₂_ref |             = 0.0
